# Chapter 7 companion notebook: Activities 2 and 3

Arabic hate speech detection: annotation agreement and evaluating a classifier against human labels.

Data: L-HSAB (Mulki et al., 2019), downloaded from its public GitHub repository. Runs on a CPU; no GPU is needed.

Run the cells in order. Step 3 needs your annotation file.

## Step 1. Load L-HSAB

In [ ]:
import csv, io, urllib.request
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score, confusion_matrix, f1_score, classification_report

DATA_URL = ("https://raw.githubusercontent.com/Hala-Mulki/"
            "L-HSAB-First-Arabic-Levantine-HateSpeech-Dataset/master/Dataset/L-HSAB")
LABELS = ["normal", "abusive", "hate"]

raw = urllib.request.urlopen(DATA_URL).read().decode("utf-8")
rows = []
for line in raw.split("\n")[1:]:              # skip header "Tweet<TAB>Class"
    parts = line.rsplit("\t", 1)
    if len(parts) == 2 and parts[1].strip() in LABELS:
        rows.append((parts[0].strip(), parts[1].strip()))
df = pd.DataFrame(rows, columns=["tweet", "label"])
print(len(df), "labeled tweets")              # expected: 5846
print(df["label"].value_counts())

## Step 2. The prepared sample

30 tweets, 10 per L-HSAB label, drawn with a fixed random seed so every class gets the same sample. Duplicate tweets are removed first.

Files written:
- `sample_student.csv`: item id and tweet only (give this to students)
- `sample_key.csv`: L-HSAB reference labels (keep for the instructor)
- `annotation_template.csv`: columns to fill in for Step 3

In [ ]:
SEED = 42
dedup = df.drop_duplicates("tweet", keep="first").reset_index(drop=True)
sample = pd.concat([dedup[dedup["label"] == lab].sample(n=10, random_state=SEED)
                    for lab in LABELS])
sample = sample.sample(frac=1, random_state=SEED).reset_index(drop=True)   # shuffle order
sample.insert(0, "item_id", [f"S{i+1:02d}" for i in range(len(sample))])

sample[["item_id", "tweet"]].to_csv("sample_student.csv", index=False, encoding="utf-8-sig")
sample[["item_id", "label"]].rename(columns={"label": "lhsab_label"}).to_csv(
    "sample_key.csv", index=False, encoding="utf-8-sig")
template = sample[["item_id", "tweet"]].copy()
for col in ["annotator_A", "annotator_B", "adjudicated", "justification"]:
    template[col] = ""
template.to_csv("annotation_template.csv", index=False, encoding="utf-8-sig")

print(sample["label"].value_counts())
sample[["item_id", "tweet"]].head()

## Step 3. Agreement between annotators (Activity 2)

In `annotation_template.csv`, each annotator labels every item independently as `hate`, `offensive`, or `neither`. After discussing disagreements, the pair enters an agreed label in `adjudicated` and a one-line reason in `justification`. Save the file as `my_annotations.csv` and upload it (Files panel on the left).

In [ ]:
ANNOTATIONS = "my_annotations.csv"
ann = pd.read_csv(ANNOTATIONS, encoding="utf-8-sig")
for col in ["annotator_A", "annotator_B", "adjudicated"]:
    ann[col] = ann[col].astype(str).str.strip().str.lower()

a, b = ann["annotator_A"], ann["annotator_B"]
p_o = (a == b).mean()
cats = sorted(set(a) | set(b))
p_e = sum((a == c).mean() * (b == c).mean() for c in cats)
kappa_manual = (p_o - p_e) / (1 - p_e)
print(f"Observed agreement p_o = {p_o:.3f}")
print(f"Chance agreement   p_e = {p_e:.3f}")
print(f"Cohen's kappa (formula) = {kappa_manual:.3f}")
print(f"Cohen's kappa (sklearn) = {cohen_kappa_score(a, b):.3f}")
ann[a != b][["item_id", "tweet", "annotator_A", "annotator_B"]]

## Step 4. Train a classifier on the rest of L-HSAB (Activity 3)

Character n-gram TF-IDF with logistic regression. The 30 sample tweets, and any duplicates of them, are excluded from training. A held-out 20% of the training data gives a reference score.

In [ ]:
train = df[~df["tweet"].isin(set(sample["tweet"]))].reset_index(drop=True)
print(len(train), "training tweets")

clf = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True),
    LogisticRegression(max_iter=2000, class_weight="balanced"))

X_tr, X_dev, y_tr, y_dev = train_test_split(
    train["tweet"], train["label"], test_size=0.2, stratify=train["label"], random_state=SEED)
clf.fit(X_tr, y_tr)
print(f"Held-out macro-F1: {f1_score(y_dev, clf.predict(X_dev), average='macro'):.3f}")
print(classification_report(y_dev, clf.predict(X_dev), digits=3))

_ = clf.fit(train["tweet"], train["label"])  # refit on all training tweets

## Step 5. Compare predictions with your adjudicated labels

The classifier uses L-HSAB labels (`normal`, `abusive`, `hate`). Your labels are mapped as: neither → normal, offensive → abusive, hate → hate. The two schemes are not identical; note this when you interpret disagreements.

In [ ]:
MAP = {"neither": "normal", "offensive": "abusive", "hate": "hate"}
ann["human"] = ann["adjudicated"].map(MAP)
if ann["human"].isna().any():
    print("Unrecognized labels in 'adjudicated':", ann.loc[ann["human"].isna(), "adjudicated"].unique())

ann["model"] = clf.predict(ann["tweet"])
key = pd.read_csv("sample_key.csv", encoding="utf-8-sig")
ann = ann.merge(key, on="item_id")

print("Model vs. your adjudicated labels (rows = human, columns = model):")
print(pd.DataFrame(confusion_matrix(ann["human"], ann["model"], labels=LABELS),
                   index=LABELS, columns=LABELS))
print(f"\nMacro-F1 against your labels: {f1_score(ann['human'], ann['model'], average='macro', labels=LABELS):.3f}")
print(f"Agreement of your labels with L-HSAB reference: {(ann['human'] == ann['lhsab_label']).mean():.2f}")

## Step 6. Examine each disagreement before counting it as a model error

For every row in `disagreements.csv`, fill in `category` with one of:
`model error`, `human annotation error`, `label-scheme difference`, `guideline ambiguity`,
and add a short note. Then summarize the patterns using the chapter's framework.

In [ ]:
dis = ann[ann["human"] != ann["model"]][
    ["item_id", "tweet", "adjudicated", "human", "model", "lhsab_label", "justification"]].copy()
dis["category"] = ""
dis["note"] = ""
dis.to_csv("disagreements.csv", index=False, encoding="utf-8-sig")
print(len(dis), "disagreements written to disagreements.csv")
dis

## Download your files

In Colab, use the Files panel, or run the cell below.

In [ ]:
try:
    from google.colab import files
    for f in ["disagreements.csv", "sample_key.csv"]:
        files.download(f)
except ImportError:
    print("Not running in Colab; files are in the working directory.")